# FPG Control Boundary Validation

## tl;dr

This notebook validates the boundary between legal obligation, upstream responsibility, and FPG runtime enforcement. KYC/AML/VASP eligibility are not FPG runtime controls.

## Context & Methods

Inputs are the control-boundary-corrected outbound matrix v2, runtime pipeline v2, BE handoff v3, and validation assertions.

### Key Assumptions

FPG starts from an Approved Transaction. It does not approve transactions, perform KYC/AML, determine VASP eligibility, custody assets, sign transactions, or guarantee settlement finality.

In [ ]:
# ruff: noqa: E501, E701, E702, I001

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'ADP-DA': ROOT = next(p for p in [ROOT, *ROOT.parents] if p.name == 'ADP-DA')
matrix = json.loads((ROOT/'03_digital_asset/artifacts/outbound_design_vNext/outbound_requirement_matrix.json').read_text(encoding='utf-8'))
pipeline = json.loads((ROOT/'03_digital_asset/artifacts/outbound_design_vNext/runtime_pipeline.json').read_text(encoding='utf-8'))
validation = json.loads((ROOT/'03_digital_asset/artifacts/outbound_design_vNext/control_boundary_validation.json').read_text(encoding='utf-8'))
req = pd.DataFrame(matrix['requirements'])
print({'requirements': len(req), 'pipeline_steps': len(pipeline['steps']), 'validation_assertions': len(validation['assertions'])})

## Data

### 1. Legal Obligation Count

In [ ]:
display(req.groupby(['evidence_type','responsibility_layer']).size().reset_index(name='count'))

## Results

### 2. Upstream Responsibility Items

In [ ]:
display(req[req['responsibility_layer'] == 'UPSTREAM_PRODUCED_DATA'][['requirement_id','required_fields','runtime_control','fpg_boundary_note']])

### 3. FPG Runtime Enforcement Items

In [ ]:
display(req[['requirement_id','required_fields','runtime_control','validation_type']])
controls = req['runtime_control'].value_counts().reset_index(); controls.columns=['runtime_control','count']; display(controls); controls.plot.bar(x='runtime_control', y='count', legend=False, color='#2f6f8f', title='FPG Runtime Control Distribution'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 4. Approved-vs-Requested Match Items

In [ ]:
display(req[req['runtime_control'] == 'APPROVED_VS_REQUESTED_MATCH'][['requirement_id','required_fields','validation_type','on_mismatch']])

### 5. Required Outbound Field

In [ ]:
display(req[req['runtime_control'] == 'REQUIRED_OUTBOUND_FIELD_PRESENCE'][['requirement_id','required_fields','on_missing','fpg_boundary_note']])

### 6. Exact / Transform / Minimize / Internal-only Distribution

In [ ]:
exact = req['required_exact'].value_counts().reset_index(); exact.columns=['required_exact','count']; display(exact); exact.plot.bar(x='required_exact', y='count', legend=False, color='#4f8f65', title='Required Exact Classification'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 7. Source System x Field Role

In [ ]:
roles = req.explode('field_roles'); display(roles.groupby(['runtime_control','field_roles']).size().unstack(fill_value=0))

### 8. Destination x Field Role

In [ ]:
dest = req.explode('destination').explode('field_roles'); display(dest.groupby(['destination','field_roles']).size().unstack(fill_value=0))

### 9. Legal Requirement -> Outbound Requirement Lineage

In [ ]:
display(req[['requirement_id','legal_basis','evidence_type','responsibility_layer','runtime_control','required_fields']])

### 10. Boundary Assertion Validation

In [ ]:
assertions = pd.DataFrame(validation['assertions']); display(assertions); print({'all_pass': (assertions['status'] == 'PASS').all()})

## Takeaways

The corrected artifacts contain no KYC/AML/VASP eligibility runtime control. FPG runtime control is limited to match, presence, exact preservation, transform/separation, destination-specific payload build, and trace binding.